In [84]:
import xml_to_midi as xm
from music21 import converter, stream, note, chord, interval, pitch, instrument

In [85]:
xml_path = 'xml_full_staves/BossaNova/JS_One_Note_Samba_a1.xml'

In [86]:
score = xm.load_score(xml_path)

part1 = score.parts[0]
part3 = score.parts[2]
part5 = score.parts[4]
part6 = score.parts[5]

In [87]:
measures1 = part1.getElementsByClass('Measure')
print(measures1[0].quarterLength)
measures5 = part5.getElementsByClass('Measure')
print(measures5[0].quarterLength)

2.0
4.0


In [88]:
score.show('t')

{0.0} <music21.text.TextBox 'One Note S...'>
{0.0} <music21.text.TextBox '2'>
{0.0} <music21.metadata.Metadata object at 0x73865f1d4e30>
{0.0} <music21.stream.Part MusicXML Part>
    {0.0} <music21.instrument.Piano 'P1: MusicXML Part: Acoustic Grand Piano'>
    {0.0} <music21.stream.Measure 0 offset=0.0>
        {0.0} <music21.layout.SystemLayout>
        {0.0} <music21.clef.TrebleClef>
        {0.0} <music21.tempo.MetronomeMark Quarter=120 (playback only)>
        {0.0} <music21.key.Key of B- major>
        {0.0} <music21.meter.TimeSignature 4/4>
        {0.0} <music21.note.Rest eighth>
        {0.5} <music21.note.Note F>
        {1.5} <music21.note.Note F>
    {2.0} <music21.stream.Measure 1 offset=2.0>
        {0.0} <music21.bar.Repeat direction=start>
        {0.0} <music21.repeat.Segno 'Segno'>
        {0.0} <music21.note.Note F>
        {1.0} <music21.note.Note F>
        {1.5} <music21.note.Note F>
        {2.5} <music21.note.Note F>
        {3.5} <music21.note.Note F>
    {6.0}

In [ ]:
def normalize_pickup_across_parts(score):
    # assume global time signature
    ts = score.recurse().getElementsByClass('TimeSignature').first()
    bar_len = ts.barDuration.quarterLength
    print('bar_len:', bar_len)

    for part in score.parts:
        m1 = part.measure(0)
        if m1 is None:
            continue

        m1_len = m1.quarterLength
        print('m1_len:', m1_len)

        if m1_len < bar_len:
            print(m1_len, bar_len)
            pickup_deficit = bar_len - m1_len
            part.shiftElements(pickup_deficit)
# end normalize_pickup_across_parts

In [90]:
normalize_pickup_across_parts(score)
score.show('t')

bar_len: 4.0
m1_len: 2.0
2.0 4.0
m1_len: 4.0
m1_len: 4.0
m1_len: 4.0
m1_len: 4.0
m1_len: 4.0
{0.0} <music21.text.TextBox 'One Note S...'>
{0.0} <music21.text.TextBox '2'>
{0.0} <music21.metadata.Metadata object at 0x73865f1d4e30>
{0.0} <music21.stream.Part MusicXML Part>
    {2.0} <music21.instrument.Piano 'P1: MusicXML Part: Acoustic Grand Piano'>
    {2.0} <music21.stream.Measure 0 offset=2.0>
        {0.0} <music21.layout.SystemLayout>
        {0.0} <music21.clef.TrebleClef>
        {0.0} <music21.tempo.MetronomeMark Quarter=120 (playback only)>
        {0.0} <music21.key.Key of B- major>
        {0.0} <music21.meter.TimeSignature 4/4>
        {0.0} <music21.note.Rest eighth>
        {0.5} <music21.note.Note F>
        {1.5} <music21.note.Note F>
    {4.0} <music21.stream.Measure 1 offset=4.0>
        {0.0} <music21.bar.Repeat direction=start>
        {0.0} <music21.repeat.Segno 'Segno'>
        {0.0} <music21.note.Note F>
        {1.0} <music21.note.Note F>
        {1.5} <music21.n

In [91]:
# find tonality before removing anything
root, mode = xm.estimate_key_from_cluster(part3)

In [92]:
print(root, mode)

<music21.note.Note B-> major


In [93]:
def earliest_harmony_offset(score, part_indices=(4, 5)):
    earliest = None

    for idx in part_indices:
        part = score.parts[idx]
        for el in part.recurse().notes:
            off = el.getOffsetInHierarchy(score)
            if earliest is None or off < earliest:
                earliest = off

    return earliest
# end earliest_harmony_offset

In [94]:
earliest = earliest_harmony_offset(score)
print(earliest)

4.0


In [95]:
def crop_score_before_offset(score, cutoff):
    for part in score.parts:
        for el in list(part.recurse().notesAndRests):
            off = el.getOffsetInHierarchy(score)
            if off < cutoff:
                print('from part: ', part)
                print('off: ', off, ' - cutoff: ', cutoff)
                print('removing: ', el)
                part.remove(el)
# end crop_score_before_offset

In [96]:
crop_score_before_offset(score, earliest)

from part:  <music21.stream.Part MusicXML Part>
off:  2.0  - cutoff:  4.0
removing:  <music21.note.Rest eighth>
from part:  <music21.stream.Part MusicXML Part>
off:  2.5  - cutoff:  4.0
removing:  <music21.note.Note F>
from part:  <music21.stream.Part MusicXML Part>
off:  3.5  - cutoff:  4.0
removing:  <music21.note.Note F>
from part:  <music21.stream.Part MusicXML Part>
off:  0.0  - cutoff:  4.0
removing:  <music21.note.Rest whole>
from part:  <music21.stream.Part Tonality>
off:  0.0  - cutoff:  4.0
removing:  <music21.note.Rest whole>
from part:  <music21.stream.Part Grouping>
off:  0.0  - cutoff:  4.0
removing:  <music21.note.Rest whole>
from part:  <music21.stream.Part MusicXML Part>
off:  0.0  - cutoff:  4.0
removing:  <music21.note.Rest whole>
from part:  <music21.stream.Part MusicXML Part>
off:  0.0  - cutoff:  4.0
removing:  <music21.note.Rest whole>


In [97]:
score.show('t')

{0.0} <music21.text.TextBox 'One Note S...'>
{0.0} <music21.text.TextBox '2'>
{0.0} <music21.metadata.Metadata object at 0x73865f1d4e30>
{0.0} <music21.stream.Part MusicXML Part>
    {2.0} <music21.instrument.Piano 'P1: MusicXML Part: Acoustic Grand Piano'>
    {2.0} <music21.stream.Measure 0 offset=2.0>
        {0.0} <music21.layout.SystemLayout>
        {0.0} <music21.clef.TrebleClef>
        {0.0} <music21.tempo.MetronomeMark Quarter=120 (playback only)>
        {0.0} <music21.key.Key of B- major>
        {0.0} <music21.meter.TimeSignature 4/4>
        {0.0} <music21.note.Rest eighth>
        {0.5} <music21.note.Note F>
        {1.5} <music21.note.Note F>
    {4.0} <music21.stream.Measure 1 offset=4.0>
        {0.0} <music21.bar.Repeat direction=start>
        {0.0} <music21.repeat.Segno 'Segno'>
        {0.0} <music21.note.Note F>
        {1.0} <music21.note.Note F>
        {1.5} <music21.note.Note F>
        {2.5} <music21.note.Note F>
        {3.5} <music21.note.Note F>
    {8.0}

In [70]:
part1 = score.parts[0]
part3 = score.parts[2]
part5 = score.parts[4]
part6 = score.parts[5]

In [68]:
part3.show('t')

{0.0} <music21.instrument.Piano 'P3: Tonality: Acoustic Grand Piano'>
{0.0} <music21.stream.Measure 0 offset=0.0>
    {0.0} <music21.layout.SystemLayout>
    {0.0} <music21.clef.TrebleClef>
    {0.0} <music21.tempo.MetronomeMark Quarter=120 (playback only)>
    {0.0} <music21.key.Key of B- major>
    {0.0} <music21.meter.TimeSignature 4/4>
    {0.0} <music21.note.Rest whole>
{4.0} <music21.stream.Measure 1 offset=4.0>
    {0.0} <music21.bar.Repeat direction=start>
    {0.0} <music21.repeat.Segno 'Segno'>
    {0.0} <music21.chord.Chord B-3 C4 D4 E-4 F4 G4 A4>
{8.0} <music21.stream.Measure 2 offset=8.0>
    {0.0} <music21.note.Rest whole>
{12.0} <music21.stream.Measure 3 offset=12.0>
    {0.0} <music21.note.Rest whole>
{16.0} <music21.stream.Measure 4 offset=16.0>
    {0.0} <music21.layout.PageLayout>
    {0.0} <music21.note.Rest whole>
{20.0} <music21.stream.Measure 5 offset=20.0>
    {0.0} <music21.note.Rest whole>
{24.0} <music21.stream.Measure 6 offset=24.0>
    {0.0} <music21.note.R

In [28]:
def extract_skyline_melody(part):
    melody = stream.Part()
    clef = part.getElementsByClass('Clef').first()
    if clef is not None:
        melody.insert(0, clef)
    ts = part.getTimeSignatures().first()
    if ts is not None:
        melody.insert(0, ts)

    # group notes by offset
    offset_dict = {}
    for n in part.flat.recurse().notes:
        print(n.offset)
        offset_dict.setdefault(n.offset, []).append(n)
    
    for offset, notes in sorted(offset_dict.items()):
        print(offset, notes)
        pitches = []
        for n in notes:
            if isinstance(n, note.Note):
                pitches.append(n)
            elif isinstance(n, chord.Chord):
                for n in n.notes:
                    pitches.append(n)
        highest = max(pitches, key=lambda n: n.pitch.midi)
        print(highest)
        melody.insert(offset, note.Note(
            highest.pitch,
            quarterLength=highest.quarterLength
        ))

    return melody
# end extract_skyline_melody

In [69]:
def extract_skyline_melody(part, score):
    melody = stream.Part()

    offset_dict = {}

    for el in part.recurse().notes:
        abs_offset = el.getOffsetInHierarchy(score)
        offset_dict.setdefault(abs_offset, []).append(el)

    for offset, notes in sorted(offset_dict.items()):
        pitches = []
        for n in notes:
            if isinstance(n, note.Note):
                pitches.append(n)
            elif isinstance(n, chord.Chord):
                pitches.extend(n.notes)

        highest = max(pitches, key=lambda n: n.pitch.midi)

        melody.insert(
            offset,
            note.Note(
                highest.pitch,
                quarterLength=highest.quarterLength
            )
        )

    return melody


In [71]:
melody = extract_skyline_melody(part1, score)

In [72]:
print(melody.show('t'))

{0.5} <music21.note.Note F>
{1.5} <music21.note.Note F>
{2.0} <music21.note.Note F>
{3.0} <music21.note.Note F>
{3.5} <music21.note.Note F>
{4.5} <music21.note.Note F>
{5.5} <music21.note.Note F>
{6.0} <music21.note.Note F>
{6.5} <music21.note.Note F>
{8.5} <music21.note.Note F>
{9.5} <music21.note.Note F>
{10.0} <music21.note.Note F>
{11.0} <music21.note.Note F>
{11.5} <music21.note.Note F>
{12.5} <music21.note.Note F>
{13.5} <music21.note.Note F>
{14.0} <music21.note.Note F>
{16.5} <music21.note.Note F>
{17.5} <music21.note.Note F>
{18.0} <music21.note.Note F>
{19.0} <music21.note.Note F>
{19.5} <music21.note.Note F>
{20.5} <music21.note.Note F>
{21.5} <music21.note.Note F>
{22.0} <music21.note.Note F>
{22.5} <music21.note.Note F>
{24.5} <music21.note.Note F>
{25.5} <music21.note.Note F>
{26.0} <music21.note.Note F>
{27.0} <music21.note.Note F>
{27.5} <music21.note.Note F>
{28.5} <music21.note.Note F>
{29.5} <music21.note.Note F>
{30.0} <music21.note.Note F>
{32.5} <music21.note.Note

In [32]:
print(root, mode)

<music21.note.Note B-> major


In [42]:
def merge_parts_parallel(part_a, part_b):
    merged = stream.Part()

    for el in part_a.flatten().notesAndRests:
        merged.insert(el.offset, el)

    for el in part_b.flatten().notesAndRests:
        merged.insert(el.offset, el)

    return merged
# end merge_parts_parallel

In [50]:
def merge_parts_parallel(part_a, part_b, score):
    merged = stream.Part()

    for el in part_a.recurse().notes:
        merged.insert(el.getOffsetInHierarchy(score), el)

    for el in part_b.recurse().notes:
        merged.insert(el.getOffsetInHierarchy(score), el)

    return merged


In [57]:
def extract_chords(part5, part6, score):
    combined = merge_parts_parallel(part5, part6, score)

    chordified = combined.chordify()

    chords = stream.Part()
    for c in chordified.recurse().getElementsByClass(chord.Chord):
        chords.insert(c.offset, chord.Chord(
            c.pitches,
            quarterLength=c.quarterLength
        ))

    return chords
# end extract_chords

In [52]:
chords = extract_chords(part5, part6, score)

In [53]:
chords.show('t')

{4.0} <music21.chord.Chord D3 F3 A3 C4 F4>
{8.0} <music21.chord.Chord D-3 F3 A-3 C-4 F4>
{12.0} <music21.chord.Chord C3 E-3 G3 B-3 F4>
{16.0} <music21.chord.Chord B2 D3 F3 A3 F4>
{20.0} <music21.chord.Chord D3 F3 A3 C4 F4>
{24.0} <music21.chord.Chord D-3 F3 A-3 C-4 F4>
{28.0} <music21.chord.Chord C3 E-3 G3 B-3 F4>
{32.0} <music21.chord.Chord B2 D3 F3 A3 F4>
{36.0} <music21.chord.Chord F2 A-2 C3 E-3 B-4>
{40.0} <music21.chord.Chord B-2 D3 F3 A-3 B-4>
{44.0} <music21.chord.Chord E-2 G2 B-2 D3 B-4>
{48.0} <music21.chord.Chord A-2 C3 E-3 G-3 B-4>
{50.0} <music21.chord.Chord A-2 C3 E-3 G-3 F4>
{52.0} <music21.chord.Chord D3 F3 A3 C4 F4>
{56.0} <music21.chord.Chord D-3 F3 A-3 C-4 F4>
{60.0} <music21.chord.Chord C3 E-3 G3 B-3 F4>
{62.0} <music21.chord.Chord B2 D3 F3 A3 F4>
{64.0} <music21.chord.Chord B-2 D3 F3 G3>
{68.0} <music21.chord.Chord E-3 G-3 B-3 D-4 B-4>
{69.0} <music21.chord.Chord E-3 G-3 B-3 D-4 D-5>
{71.0} <music21.chord.Chord E-3 G-3 B-3 D-4 B-4>
{72.0} <music21.chord.Chord A-2 C3